In [3]:
import pandas as pd
import numpy as np
import json
import os
import pickle
import faiss
from sentence_transformers import SentenceTransformer
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import PyPDF2
import docx
import re
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

In [6]:
# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


In [19]:
# Load dataset
df = pd.read_csv("AI_Resume_Matcher_Dataset.csv")
print(f"Loaded {len(df)} resumes")
print(f"Columns: {df.columns.tolist()}")
print(f"Sample data:\n{df.head(2)}")
print(f"\nDataset info:")
print(df.info())

Loaded 2000 resumes
Columns: ['Name', 'Experience_Years', 'Skills', 'Education', 'Applied_Job_Role']
Sample data:
          Name  Experience_Years             Skills Education Applied_Job_Role
0  Candidate_0                 1   Node.js, MongoDB       BCA   Data Scientist
1  Candidate_1                 7  JavaScript, React     B.Com  Product Manager

Dataset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Name              2000 non-null   object
 1   Experience_Years  2000 non-null   int64 
 2   Skills            2000 non-null   object
 3   Education         2000 non-null   object
 4   Applied_Job_Role  2000 non-null   object
dtypes: int64(1), object(4)
memory usage: 78.3+ KB
None


In [20]:
# Check for missing values
print(f"\nMissing values:\n{df.isnull().sum()}")


Missing values:
Name                0
Experience_Years    0
Skills              0
Education           0
Applied_Job_Role    0
dtype: int64


TEXT EXTRACTION FUNCTIONS (CRITICAL FOR HANDLING DIFFERENT FILE TYPES)

In [28]:
# Text extraction from different formats
def extract_text_from_file(file_path):
  if file_path.endswith(".pdf"):
    with open(file_path, "rb") as file:
      reader = PyPDF2.PdfReader(file)  #read pages and text
      text = ""
      for page in reader.pages:
        text += page.extract_text()
    return text
  elif file_path.endswith(".docx"):
    doc = docx.Document(file_path)
    text = "\n".join([paragraph.text for paragraph in doc.paragraphs])
    return text
  else:
      with open(file_path, "r", encoding="utf-8", errors="ignore") as file:
        return file.read()

TEXT PREPROCESSING FUNCTIONS

In [42]:
# Preprocess and create resume text from structured data
def create_resume_text(row):
  """Create a comprehensive resume text from structured data"""
  resume_parts = []

  # Name
  if pd.notna(row.get("Name", "")):
    resume_parts.append(f"Name: {row["Name"]}")

  # Experience
  if pd.notna(row.get("Experience_Years", "")):
    resume_parts.append(f"Experience: {row['Experience_Years']} Years")

  # Skills
  if pd.notna(row.get("Skills", "")):
    skills = row["Skills"]
    if isinstance(skills, str):
      # Split skills if they're comma-separated
      skill_list = [s.strip() for s in skills.split(",")]
      resume_parts.append(f"Skills: {', '.join(skill_list)}")

  # Education
  if pd.notna(row.get("Education", "")):
    resume_parts.append(f"Education: {row["Education"]}")

  # Applied job Role
  if pd.notna(row.get("Applied_Job_Role", "")):
    resume_parts.append(f"Applied for: {row['Applied_Job_Role']}")

  return " ".join(resume_parts) if resume_parts else "No resume data available"


In [43]:
# Create processed resumes
df["Resume_Text"] = df.apply(create_resume_text, axis=1)

In [57]:
# Preprocess resume text
def preprocess_text(text):
  # remove extra whitepace
  text = re.sub(r"\s", " ", text).strip()

  # Remove special characters (keep only basic punctuation)
  text = re.sub(r"[^\w\s\.\,\-\:\(\)]", " ", text)

  return text

In [58]:
df["Processed_Text"] = df["Resume_Text"].apply(preprocess_text)

In [59]:
# create resume chunks for better retrieval
def chunk_resume(text, chunk_size=500, ovarlap=50):
  """
  Split resume text into overlaping chunks for better retrieval
  """
  if not text or len(text.strip()) == 0:
    return ["No content available"]

  words = text.split()
  chunks = []

  if len(words) == 0:
    return [text]

  for i in range(0, len(words), chunk_size - ovarlap):
    chunk = " ".join(words[i:i + chunk_size])
    if chunk.strip():
      chunks.append(chunk)
  # if no chunk were created, return the original text
  if not chunks:
    chunks = [text[:500]]

  return chunks

PROCESS RESUMES FROM DATASET

In [60]:
resume_data = []
for idx, row in tqdm(df.iterrows(), total=len(df)):
  # extract and preprocess text
  resume_text = row["Processed_Text"]

  # Create chunks
  chunks = chunk_resume(resume_text)

  resume_data.append({
      "id": idx,
      "name": row.get("Name", ""),
      "experience": row.get("Experience_Years", 0),
      "skills": row.get("Skills", ""),
      "education": row.get("Education", ""),
      "applied_job_role": row.get("Applied_job_title", ""),
      "full_text": resume_text,
      "chunks": chunks,
      "num_chunks": len(chunks),
      "metadata": {
          "name": row.get("Name", f"Candidate_{idx}"),
          "experience": row.get("Experience_Years", 0),
          "skills": row.get("Skills", ""),
          "education": row.get("Education", ""),
          "applied_jo_role": row.get("Applied_Job_Role", ""),
          "file_name": f"resume_{idx}.txt"
      }
  })

print(f"\nProcessed {len(resume_data)} resumes with {sum(r["num_chunks"] for r in resume_data)} total_chunks")

100%|██████████| 2000/2000 [00:00<00:00, 11805.93it/s]


Processed 2000 resumes with 2000 total_chunks
